In [191]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
import librosa
from sklearn.pipeline import make_pipeline
from scipy.stats import skew, kurtosis
from scipy.signal import hilbert
from math import log10
import parselmouth
from maad import sound as suono
from maad import features as ft

In [146]:
csvTotale = pd.read_csv('.././data/development.csv', header=0, index_col=0).drop(columns=['sampling_rate'])
csvTotale['gender'] = csvTotale['gender'].map(lambda x: 0 if x=='male' else 1)
csvTotale['tempo'] = csvTotale['tempo'].map(lambda x: float(x[1:-1]))
csvTotale['path'] = csvTotale['path'].map(lambda x: x.split('/')[-1])

In [147]:
csvEval = pd.read_csv('.././data/evaluation.csv', header=0, index_col=0).drop(columns=['sampling_rate'])
csvEval['gender'] = csvEval['gender'].map(lambda x: 0 if x=='male' else 1)
csvEval['tempo'] = csvEval['tempo'].map(lambda x: float(x[1:-1]))
csvEval['path'] = csvEval['path'].map(lambda x: x.split('/')[-1])

In [ ]:
"""(cross_val_score(HistGradientBoostingRegressor(categorical_features=csvTotale.drop(columns=['path', 'age']).dtypes == 'object'), 
                csvTotale.drop(columns=['path', 'age']), 
                csvTotale['age'], cv=10, scoring='neg_mean_absolute_error').mean(),
 cross_val_score(MLPRegressor(), 
                csvTotale.drop(columns=['path', 'age', 'ethnicity']), 
                csvTotale['age'], cv=10, scoring='neg_mean_absolute_error').mean())"""

# BASELINE = -7.24, -8.18

In [149]:
audioDevelopment = {file: librosa.load('.././data/audios_development/'+file, sr=22050)[0] for file in csvTotale['path']}

In [237]:
def getMFCC(audio):
    numFcc = 35
    return pd.Series(librosa.feature.mfcc(y=audio, sr=22050, n_mfcc=numFcc).mean(axis=1), 
                     index=[f'mfcc{i}' for i in range(numFcc)])

In [238]:
def getSpectralEnergy(audio):
    S = librosa.stft(audio)
    freqs = librosa.fft_frequencies(sr=22050)
    return pd.Series([np.sum(np.abs(S[(freqs >= 250) & (freqs <= 650)])**2), np.sum(np.abs(S[(freqs >= 1000) & (freqs <= 8000)])**2)],
                        index=['spectralEnergy250-650', 'spectralEnergy1000-8000'])

In [239]:
def computeF0(audio):
    sound = parselmouth.Sound(audio)
    pitch = sound.to_pitch()
    info = str(pitch.info()).split('\n')
    return pd.Series([pitch.count_voiced_frames(), pitch.get_mean_absolute_slope(), pitch.xmax-pitch.xmin, 
                      pitch.n_frames, *[float(info[15+i].split('=')[2].lstrip().split(' =')[0].split()[0]) for i in range(0, 5)],
                      *[float(info[21+i].split('=')[2].lstrip().split(' =')[0].split()[0]) for i in range(0, 3)]],
                     index=['nVoicedFrames', 'meanAbsoluteSlope', 'duration', 'nFrames', 
                            'q10', 'q16', 'q50', 'q84', 'q90', '84-median', 'median-16', '90-10']
                     )

In [240]:
def computeMath(audio):
    return pd.Series([skew(audio), kurtosis(audio), np.mean(np.abs(hilbert(audio))), np.mean(np.abs(np.fft.fft(audio)))],
                     index=['skew', 'kurtosis', 'hilbertMean', 'fftMean'])

In [241]:
def computSNR(audio):
    return pd.Series([suono.temporal_snr(audio)[-1]],
                        index=['temporalSNR'])

In [242]:
def computeTemporalMedia(audio):
    return pd.Series([ft.temporal_median(audio)],
                     index=['temporalMedian'])

In [243]:
def computeAllFeatures(audio):
    return pd.Series(ft.all_temporal_features(audio, fs=22050, nperseg=256).values[0],
                     index=['sm', 'sv', 'ss', 'sk', 'Time 5%', "Time 25%", "Time 50%", "Time 75%", "Time 95%  ", 
                            "zcr", "duration_5", "duration_90"])

In [244]:
def computePeakFrequency(audio):
    return pd.Series(ft.peak_frequency(audio, fs=22050, nperseg=256, amp=True),
                     index=['peakFrequency', 'peakFrequencyAmp'])

In [245]:
def computeMetrics(audios):
    return pd.DataFrame({file: pd.concat([
            getMFCC(audios[file]),
            getSpectralEnergy(audios[file]),
            computeF0(audios[file]),
            computeMath(audios[file]),
            computSNR(audios[file]),
            computeTemporalMedia(audios[file]),
            computeAllFeatures(audios[file]),
            computePeakFrequency(audios[file])
            ], axis=0)
        for file in audios}).T

In [ ]:
metrics = pd.concat([computeMetrics(audioDevelopment), 
                    csvTotale.set_index('path')[['hnr', 'shimmer', 'jitter', 'gender', 'max_pitch', 'mean_pitch', 'min_pitch']]], axis=1)

metrics['log_hnr']= metrics['hnr'].map(lambda x: 20*log10(np.abs(x)))

In [262]:
temp.columns

Index(['mfcc0', 'mfcc1', 'mfcc2', 'mfcc3', 'mfcc4', 'mfcc5', 'mfcc6', 'mfcc7',
       'mfcc8', 'mfcc9', 'mfcc10', 'mfcc11', 'mfcc12', 'mfcc13', 'mfcc14',
       'mfcc15', 'mfcc16', 'mfcc17', 'mfcc18', 'mfcc19', 'mfcc20', 'mfcc21',
       'mfcc22', 'mfcc23', 'mfcc24', 'mfcc25', 'mfcc26', 'mfcc27', 'mfcc28',
       'mfcc29', 'mfcc30', 'mfcc31', 'mfcc32', 'mfcc33', 'mfcc34',
       'spectralEnergy250-650', 'spectralEnergy1000-8000', 'nVoicedFrames',
       'meanAbsoluteSlope', 'duration', 'nFrames', 'q10', 'q16', 'q50', 'q84',
       'q90', '84-median', 'median-16', '90-10', 'skew', 'kurtosis',
       'hilbertMean', 'fftMean', 'temporalSNR', 'temporalMedian', 'sm', 'sv',
       'ss', 'sk', 'Time 5%', 'Time 25%', 'Time 50%', 'Time 75%', 'Time 95%  ',
       'zcr', 'duration_5', 'duration_90', 'peakFrequency', 'peakFrequencyAmp',
       'hnr', 'shimmer', 'jitter', 'gender', 'max_pitch', 'mean_pitch',
       'min_pitch', 'log_hnr', 'age'],
      dtype='object')

In [263]:
cross_val_score(HistGradientBoostingRegressor(), temp.drop(columns=['age']),
                temp['age'], cv=15, scoring='neg_mean_absolute_error').mean()

np.float64(-6.678625782591936)

In [266]:
for item, imp in sorted(zip(list(metrics.columns)+['age'], 
                            pd.concat([metrics, csvTotale.set_index(csvTotale['path'])['age']], axis=1).corr('spearman')['age']), key=lambda x: abs(x[1]), reverse=True):
    print(f'{item}: {imp}')

age: 1.0
nFrames: 0.6022603562876916
duration: 0.602252908183809
Time 95%  : 0.6016573045219915
duration_90: 0.5994765605409939
duration_5: 0.5948130345006974
Time 75%: 0.5904688371671031
Time 50%: 0.5818011133930301
nVoicedFrames: 0.5644983266317832
Time 25%: 0.5518922891861292
Time 5%: 0.5350617896196377
hnr: -0.5326292343962696
log_hnr: 0.5320784492500874
spectralEnergy250-650: 0.5303528487661241
sm: 0.5010898324353773
fftMean: 0.49037696789958823
spectralEnergy1000-8000: 0.4742084761261652
max_pitch: 0.4694639426920942
min_pitch: -0.46452284591811516
temporalSNR: 0.4389244252671011
mfcc6: -0.4222780140021344
ss: -0.41707174962926924
skew: -0.4170666241622713
mean_pitch: 0.36619218880953386
zcr: 0.3610859442341537
jitter: 0.34920179676737306
mfcc13: -0.34195572156396453
mfcc33: -0.32793932338185033
kurtosis: 0.32287861765139875
sk: 0.3228641469774155
mfcc15: -0.3187933132261265
mfcc17: -0.31683729202774413
mfcc31: -0.2983493012563597
mfcc2: -0.28005733072060096
mfcc11: -0.2720810656

In [267]:
audioEval = {file: librosa.load('.././data/audios_evaluation/'+file, sr=22050)[0] for file in csvEval['path']}

In [268]:
metricsEval = pd.concat([computeMetrics(audioEval), 
                    csvEval.set_index('path')[['hnr', 'shimmer', 'jitter', 'gender', 'max_pitch', 'mean_pitch', 'min_pitch']]], axis=1)
metricsEval['log_hnr']= metricsEval['hnr'].map(lambda x: 20*log10(np.abs(x)))

c:\Users\utente\OneDrive\Desktop\provaLibrosa\.venv\Lib\site-packages\maad\util\miscellaneous.py:413: RuntimeWarning: divide by zero encountered in log10
  y = 10*log10(x)   # take log
c:\Users\utente\OneDrive\Desktop\provaLibrosa\.venv\Lib\site-packages\maad\util\miscellaneous.py:413: RuntimeWarning: divide by zero encountered in log10
  y = 10*log10(x)   # take log
c:\Users\utente\OneDrive\Desktop\provaLibrosa\.venv\Lib\site-packages\maad\util\miscellaneous.py:413: RuntimeWarning: divide by zero encountered in log10
  y = 10*log10(x)   # take log
c:\Users\utente\OneDrive\Desktop\provaLibrosa\.venv\Lib\site-packages\maad\util\miscellaneous.py:413: RuntimeWarning: divide by zero encountered in log10
  y = 10*log10(x)   # take log
c:\Users\utente\OneDrive\Desktop\provaLibrosa\.venv\Lib\site-packages\maad\util\miscellaneous.py:413: RuntimeWarning: divide by zero encountered in log10
  y = 10*log10(x)   # take log
c:\Users\utente\OneDrive\Desktop\provaLibrosa\.venv\Lib\site-packages\maad\

In [269]:
pd.Series(HistGradientBoostingRegressor().fit(metrics, csvTotale['age']).predict(metricsEval),
          name='Predicted', index=csvEval.index).sort_index().to_csv('predictedBeforeNanna.csv', header=True)